# HAR Preprocessing Notebook
This notebook performs signal standardization, temporal windowing, Butterworth filtering, feature extraction, normalization, and exports scaling parameters for the UCI HAR dataset.

The reason we do the preprocessing in this step is because raw data from the train.csv and test.csv datasets is noisy and high-dimensional. This means the ML cannot learn from it directly. Trash in = trash out. We have to transform them into meaningful features.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from scipy.signal import butter, lfilter
import json
import os

In [ ]:
# Load data from working directory
# Assumes train.csv and test.csv are in the same directory as this notebook
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

In [5]:
# explore dfs 
print('Train DataFrame shape:', train_df.shape)
print('Test DataFrame shape:', test_df.shape)
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
train_df.head(), test_df.head()

Train DataFrame shape: (7352, 563)
Test DataFrame shape: (2947, 563)
Train columns: ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z', 'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z', 'tBodyAcc-max()-X', 'tBodyAcc-max()-Y', 'tBodyAcc-max()-Z', 'tBodyAcc-min()-X', 'tBodyAcc-min()-Y', 'tBodyAcc-min()-Z', 'tBodyAcc-sma()', 'tBodyAcc-energy()-X', 'tBodyAcc-energy()-Y', 'tBodyAcc-energy()-Z', 'tBodyAcc-iqr()-X', 'tBodyAcc-iqr()-Y', 'tBodyAcc-iqr()-Z', 'tBodyAcc-entropy()-X', 'tBodyAcc-entropy()-Y', 'tBodyAcc-entropy()-Z', 'tBodyAcc-arCoeff()-X,1', 'tBodyAcc-arCoeff()-X,2', 'tBodyAcc-arCoeff()-X,3', 'tBodyAcc-arCoeff()-X,4', 'tBodyAcc-arCoeff()-Y,1', 'tBodyAcc-arCoeff()-Y,2', 'tBodyAcc-arCoeff()-Y,3', 'tBodyAcc-arCoeff()-Y,4', 'tBodyAcc-arCoeff()-Z,1', 'tBodyAcc-arCoeff()-Z,2', 'tBodyAcc-arCoeff()-Z,3', 'tBodyAcc-arCoeff()-Z,4', 'tBodyAcc-correlation()-X,Y', 'tBodyAcc-correlation()-X,Z', 'tBodyAcc-correlation

(   tBodyAcc-mean()-X  tBodyAcc-mean()-Y  tBodyAcc-mean()-Z  tBodyAcc-std()-X  \
 0           0.288585          -0.020294          -0.132905         -0.995279   
 1           0.278419          -0.016411          -0.123520         -0.998245   
 2           0.279653          -0.019467          -0.113462         -0.995380   
 3           0.279174          -0.026201          -0.123283         -0.996091   
 4           0.276629          -0.016570          -0.115362         -0.998139   
 
    tBodyAcc-std()-Y  tBodyAcc-std()-Z  tBodyAcc-mad()-X  tBodyAcc-mad()-Y  \
 0         -0.983111         -0.913526         -0.995112         -0.983185   
 1         -0.975300         -0.960322         -0.998807         -0.974914   
 2         -0.967187         -0.978944         -0.996520         -0.963668   
 3         -0.983403         -0.990675         -0.997099         -0.982750   
 4         -0.980817         -0.990482         -0.998321         -0.979672   
 
    tBodyAcc-mad()-Z  tBodyAcc-max()-X  ..

### Preprocessing

We have to start with standardization to make sure our data is on a similar scale, which is important for most algorithms, especially NNs. It will also remove biases from the data.

In [ ]:
# Drop non-feature columns
feature_columns = [col for col in train_df.columns if col not in ['Activity', 'subject']]
X_train = train_df[feature_columns].values
y_train = train_df['Activity'].values
X_test = test_df[feature_columns].values
y_test = test_df['Activity'].values

print(f"Features shape: {X_train.shape}")
print(f"Number of features: {len(feature_columns)}")

# Standardization using MinMaxScaler (preferred for neural networks)
# Scale to [-1, 1] range to prevent explosive weight growth
scaler = MinMaxScaler(feature_range=(-1, 1))

# CRITICAL: Fit ONLY on training data to avoid data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaling parameters
scaler_params = {
    'min': scaler.data_min_.tolist(),
    'scale': scaler.scale_.tolist(),
    'feature_names': feature_columns
}

print(f"Scaled training shape: {X_train_scaled.shape}")
print(f"Training data range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")

Feature scaling parameters saved to scaler_params.json


In [ ]:
# Create DataFrames with scaled features
train_df_scaled = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
)
train_df_scaled['Activity'] = y_train
train_df_scaled['subject'] = train_df['subject'].values

test_df_scaled = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns
)
test_df_scaled['Activity'] = y_test
test_df_scaled['subject'] = test_df['subject'].values

print("✓ Scaled training data prepared")
print(f"  Shape: {train_df_scaled.shape}")
print(f"  Columns: Activity={len(np.unique(y_train))} classes, Features={len(feature_columns)}")

Train set: (5881, 561)
Validation set: (1471, 561)


### Now x_train will be our train set, X_val will be validation set, and X_test_scaled will be the test set.

In [ ]:
# Save processed datasets and scaling parameters
train_df_scaled.to_csv('train_processed.csv', index=False)
test_df_scaled.to_csv('test_processed.csv', index=False)

# Save scaler parameters for use in the web app
with open('scaler_params.json', 'w') as f:
    # Store in sklearn format + custom format for TypeScript
    sklearn_scaler = {
        'min': scaler.data_min_.tolist(),
        'scale': scaler.scale_,
        'feature_names': feature_columns
    }
    json.dump(sklearn_scaler, f, indent=2)

print("✓ Saved processed data:")
print(f"  train_processed.csv ({train_df_scaled.shape})")
print(f"  test_processed.csv ({test_df_scaled.shape})")
print(f"  scaler_params.json")

In [7]:

# we use butterworth low-pass filter to remove high-frequency noise from the sensor data,
# which can improve the performance of the model by focusing on the relevant signal patterns for human activity recognition.

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = lfilter(b, a, data) 
    return y

# Parameters
fs = 50  # Sampling frequency
cutoff = 0.3  # Cutoff frequency
window_size = 128
step_size = window_size // 2



filtered_train = np.apply_along_axis(butter_lowpass_filter, 0, X_train_scaled, cutoff, fs)
filtered_test = np.apply_along_axis(butter_lowpass_filter, 0, X_test_scaled, cutoff, fs)


# Windowing

def create_windows(data, window_size, step_size):
    return np.array([data[i:i+window_size] for i in range(0, data.shape[0] - window_size + 1, step_size)])

train_windows = create_windows(filtered_train, window_size, step_size)
test_windows = create_windows(filtered_test, window_size, step_size)

def assign_window_labels(labels, window_size, step_size):
    return [labels[i:i+window_size].mode()[0] for i in range(0, len(labels) - window_size + 1, step_size)]

train_labels = train_df['Activity']
test_labels = test_df['Activity']
train_window_labels = assign_window_labels(train_labels, window_size, step_size)
test_window_labels = assign_window_labels(test_labels, window_size, step_size)

In [ ]:
# Final verification
print("\n" + "="*60)
print("PREPROCESSING COMPLETE")
print("="*60)
print(f"\nActivity Distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for activity, count in zip(unique, counts):
    print(f"  {activity}: {count} samples")

print(f"\nData Statistics:")
print(f"  Training samples: {X_train_scaled.shape[0]}")
print(f"  Test samples: {X_test_scaled.shape[0]}")
print(f"  Features per sample: {X_train_scaled.shape[1]}")
print(f"  Feature range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"  Feature mean: {X_train_scaled.mean():.4f}")
print(f"  Feature std: {X_train_scaled.std():.4f}")

print("\nFiles saved:")
print(f"  ✓ train_processed.csv")
print(f"  ✓ test_processed.csv")
print(f"  ✓ scaler_params.json")
print("\nReady for training! Run: python har_training.py --train_path train.csv --test_path test.csv")